# 02 - Quality Control

This notebook performs sample-level and feature-level QC diagnostics.

## Script equivalent

The same workflow is available via `scripts/02_quality_control.py`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from config import (
    DATA_PROCESSED_DIR,
    FINAL_FORMATTED_FILENAME,
    FIGURES_DIR,
    TABLES_DIR,
    ensure_project_dirs,
)
from src.data_utils import get_lipid_columns
from src.stats_utils import run_lof_outlier_detection, run_shapiro_normality_tests

ensure_project_dirs()


## Step 1: Load processed analysis dataset


In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR / FINAL_FORMATTED_FILENAME)
lipid_cols = get_lipid_columns(df)
print("Dataset shape:", df.shape)
print("Lipid columns:", len(lipid_cols))


## Step 2: Outlier detection with Local Outlier Factor (LOF)


In [ ]:
lof_df = run_lof_outlier_detection(df, lipid_columns=lipid_cols, n_neighbors=100)
lof_df.to_csv(TABLES_DIR / "qc_lof_scores.csv", index=False)
print("LOF rows:", len(lof_df))
lof_df.head()


## Step 3: Shapiro-Wilk normality tests with FDR correction


In [ ]:
normality_df = run_shapiro_normality_tests(df, lipid_columns=lipid_cols)
normality_df.to_csv(TABLES_DIR / "qc_shapiro_normality.csv", index=False)
print("Normality rows:", len(normality_df))
normality_df.head(10)


## Step 4: Save QC diagnostic plots


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(lof_df["lof_score"], bins=30, edgecolor="black")
plt.title("LOF Score Distribution")
plt.xlabel("Negative Outlier Factor")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "qc_lof_score_distribution.png", dpi=220)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(normality_df["p_value"], bins=30, edgecolor="black")
plt.title("Shapiro-Wilk P-value Distribution")
plt.xlabel("P-value")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "qc_shapiro_pvalue_distribution.png", dpi=220)
plt.show()


## Next notebook

Run `notebooks/03_statistical_analysis.ipynb` for association models and sex-interaction ANCOVA.
